# Sistema de Recomendação de Hotéis — Documentação do Modelo

**Introdução à Inteligência Artificial — Trabalho 1**

**Integrantes**
- Artur Kohara Guerra — matrícula 231025181  
- Celio Junio de Freitas Eduardo — matrícula 211010350  
- Rafael De Lima Pereira — matrícula 242043277  

Este notebook documenta a engenharia do modelo iterativo, detalhando a transição de um modelo de Feedback Implícito para um **Espaço Vetorial de Feedback Explícito Multicritério (10 Dimensões)**. Execute as células **Python** na ordem (Python 3.10+; `pip install -r requirements.txt`).

> **Aviso:** A interface web (Streamlit) não deve ser iniciada dentro do Jupyter — ela roda num terminal à parte (`streamlit run app.py`).

## 1. Arquitetura de Dados: Espaço Vetorial de 10 Dimensões

Para contornar as limitações de agregação de informações escalares, o modelo mapeia hotéis e percepções de usuários em $\mathbf{v} \in \mathbb{R}^{10}$.
Os atributos analisados são: `luxo`, `lazer`, `urbano`, `pet_friendly`, `kids_friendly`, `acessibilidade`, `seguranca`, `preco`, `silencio`, e `capacidade`.

A célula abaixo demonstra como os usuários sintéticos (`data_generator.py`) projetam a percepção dessas características com ruído gaussiano — simulando subjetividade humana. Esse mecanismo gera o `ratings_vector` que, com a coluna `logica_geracao = 'Perfil+Região+Tradeoff'`, alimenta a matriz de utilidade do banco de dados.

In [ ]:
import numpy as np
import pandas as pd
import math

FEATURES = ['luxo', 'lazer', 'urbano', 'pet_friendly', 'kids_friendly', 'acessibilidade', 'seguranca', 'preco', 'silencio', 'capacidade']
np.random.seed(42)

# --- Geração de Percepção Multicritério Sintética (data_generator.py) ---
# Replicação exata do trecho 80% do gerador (logica = 'Perfil+Região+Tradeoff')
def gerar_ratings_sinteticos(features_hotel_01):
    """
    Converte features do hotel em [0,1] para notas de usuário em [1,5]
    adicionando ruído gaussiano (std=0.5) para simular subjetividade.
    Espelha o código de data_generator.py:
        ratings_vector = np.clip(np.round(features_val * 4 + 1 + ruido_percepcao), 1, 5)
    """
    ruido_percepcao = np.random.normal(0, 0.5, size=len(FEATURES))
    ratings_vector = np.clip(np.round(features_hotel_01 * 4.0 + 1.0 + ruido_percepcao), 1, 5)
    return ratings_vector

# Simulando um hotel predominantemente luxuoso e silencioso
hotel_exemplo = np.array([0.9, 0.4, 0.2, 0.1, 0.1, 0.5, 0.8, 0.1, 0.9, 0.3])
ratings_sinteticos = gerar_ratings_sinteticos(hotel_exemplo)

print("Features reais do hotel (Normalizadas [0, 1]):")
print({f: round(val, 2) for f, val in zip(FEATURES, hotel_exemplo)})
print("\nRatings gerados pelo usuário sintético (Escala [1, 5]):")
print({f: int(val) for f, val in zip(FEATURES, ratings_sinteticos)})

## 2. Perfil do Usuário e Vetor Final de Busca

Antes de executar qualquer algoritmo de recomendação, `RecomendacaoController.iniciar_sessao()` constrói o **vetor de busca** em três etapas:

### 2.1 Vetor de Contexto (`_build_context_vector`)
Mapeia as respostas do formulário (tipo de viagem, região, necessidades especiais) diretamente para $\mathbf{v}_{ctx} \in [0, 1]^{10}$. Sem histórico, esse vetor é usado isoladamente.

### 2.2 Perfil Latente do Usuário (`_build_user_profile`)
Dado o histórico de avaliações do usuário (10 colunas de notas em $[1, 5]$), o perfil é calculado em duas etapas:
1. **Normalização**: $w_i = \frac{nota_i - 1}{4}$ → projeta cada nota para $[0, 1]$
2. **Agregação**: $\mathbf{w}_{perfil} = \text{nanmean}(W_{\text{histórico}}, \text{axis}=0)$ → vetor médio das interações

### 2.3 Alpha-Blending (Fusão Contexto + Perfil)
O vetor final que alimenta KNN e FM combina os dois com peso $\alpha = 0.85$, priorizando o contexto imediato mas preservando o perfil histórico:

$$\mathbf{v}_{final} = 0.85 \cdot \mathbf{v}_{ctx} + 0.15 \cdot \mathbf{w}_{perfil}$$

In [ ]:
# --- Replicação exata de _build_user_profile (recomendacao_controller.py) ---

def build_user_profile(historico_notas_df):
    """
    Recebe DataFrame [N x 10] de notas em [1,5].
    Normaliza para [0,1] e retorna o vetor médio do usuário.
    """
    matriz_notas = historico_notas_df.values.astype(float)
    matriz_normalizada = (matriz_notas - 1.0) / 4.0  # [1,5] -> [0,1]
    perfil_medio = np.nanmean(matriz_normalizada, axis=0)
    return np.nan_to_num(perfil_medio, nan=0.1)  # trata colunas vazias

# Simulação: histórico com 3 avaliações (notas em [1,5])
historico_simulado = pd.DataFrame([
    [5, 3, 1, 1, 1, 3, 5, 1, 5, 2],
    [4, 2, 1, 1, 1, 4, 4, 1, 4, 2],
    [5, 3, 2, 1, 1, 3, 5, 1, 5, 3],
], columns=FEATURES)

perfil_usuario = build_user_profile(historico_simulado)
print("Perfil normalizado do usuário [0, 1]:")
print({f: round(v, 3) for f, v in zip(FEATURES, perfil_usuario)})

# Alpha-blending: contexto (negócios) + perfil histórico
ALPHA = 0.85
vetor_contexto_negocios = np.array([0.1, 0.0, 1.0, 0.1, 0.1, 0.1, 0.1, 0.8, 0.9, 0.1])

vetor_final = ALPHA * vetor_contexto_negocios + (1 - ALPHA) * perfil_usuario
print("\nVetor final de busca (alpha-blending, alpha=0.85) [0, 1]:")
print({f: round(v, 3) for f, v in zip(FEATURES, vetor_final)})

## 3. Algoritmos de Ordenação: KNN e FM

A controladora opera dois algoritmos de ranqueamento com o mesmo `vetor_final` de entrada. Ambos recebem a **matriz de hotéis** $\mathbf{F} \in [0,1]^{N \times 10}$ filtrada pela região selecionada.

### 3.1 K-Nearest Neighbors (`_computar_knn`)
Mede a **similaridade de cosseno** entre o vetor de busca e cada hotel:

$$\text{sim}(\mathbf{v}_{final}, \mathbf{f}_h) = \frac{\mathbf{v}_{final} \cdot \mathbf{f}_h}{\|\mathbf{v}_{final}\| \cdot \|\mathbf{f}_h\| + \varepsilon}$$

onde $\varepsilon = 10^{-9}$ evita divisão por zero. Hotéis são ordenados por similaridade decrescente. O score está em $[-1, 1]$; scores maiores = maior alinhamento angular.

### 3.2 Factorization Machines (`_computar_fm`)
Estima a **utilidade bruta** via produto escalar e penaliza interações latentes de conflito:

$$\text{score}_{FM}(h) = \mathbf{v}_{final} \cdot \mathbf{f}_h - \lambda \cdot f_h^{\text{luxo}} \cdot f_h^{\text{urbano}}$$

onde $\lambda = 0.6$ é a penalidade do conflito *Luxo × Urbano* (hotéis de alto luxo em zonas barulhentas). O score é **não normalizado** — usado exclusivamente para ranqueamento relativo via `argsort`.

> **Nota:** A paginação de ambos os algoritmos é implementada por `offset` e `limit` sobre os índices já ordenados, evitando reordenação a cada carregamento.

In [ ]:
# --- Replicação exata de _computar_knn e _computar_fm (recomendacao_controller.py) ---

# Matriz simulada de hotéis [N x 10] em [0, 1]
np.random.seed(7)
N_HOTEIS = 6
matriz_hoteis = np.random.beta(2, 2, (N_HOTEIS, 10))
ids_hoteis = [f'H{i:02d}' for i in range(N_HOTEIS)]

def computar_knn(vetor_busca, matriz_h, ids, offset=0, limit=3):
    """Similaridade de cosseno vetorizada — espelha _computar_knn."""
    dot_products = np.dot(matriz_h, vetor_busca)
    norm_hoteis  = np.linalg.norm(matriz_h, axis=1)
    norm_busca   = np.linalg.norm(vetor_busca)
    similaridades = dot_products / (norm_hoteis * norm_busca + 1e-9)

    indices_ordenados = np.argsort(similaridades)[::-1]
    pagina = indices_ordenados[offset : offset + limit]
    return [(ids[i], round(similaridades[i], 4)) for i in pagina]

def computar_fm(vetor_busca, matriz_h, ids, lambda_p=0.6, offset=0, limit=3):
    """Produto escalar bruto com penalidade Luxo×Urbano — espelha _computar_fm."""
    scores_base  = np.dot(matriz_h, vetor_busca)
    penalidades  = lambda_p * (matriz_h[:, 0] * matriz_h[:, 2])  # col0=luxo, col2=urbano
    scores_finais = scores_base - penalidades

    indices_ordenados = np.argsort(scores_finais)[::-1]
    pagina = indices_ordenados[offset : offset + limit]
    return [(ids[i], round(scores_finais[i], 4)) for i in pagina]

print("KNN — Top 3 por similaridade de cosseno:")
for hotel_id, score in computar_knn(vetor_final, matriz_hoteis, ids_hoteis):
    print(f"  {hotel_id}: cosine_sim = {score}")

print("\nFM  — Top 3 por utilidade bruta (dot - penalidade):")
for hotel_id, score in computar_fm(vetor_final, matriz_hoteis, ids_hoteis):
    print(f"  {hotel_id}: fm_score = {score}")

## 4. Avaliação de Performance (Métricas Globais)

As métricas são calculadas ao fim de cada sessão por `_gerar_dashboard_metricas_globais()`, consumindo o histórico completo de avaliações do banco.

### 4.1 RMSE — Avalia o FM
Filtra avaliações com `logica_geracao IN ('FM', 'Perfil+Região+Tradeoff')` (sessões FM orgânicas + dados sintéticos do gerador). Para cada linha, computa a **predição real do FM** via `_obter_predicao_fm_real`:

1. Recupera $\mathbf{w}_{usuario} = $ `_build_user_profile(histórico)` → vetor normalizado $[0,1]$  
2. Produto escalar: $\text{dot} = \mathbf{w}_{usuario} \cdot \mathbf{f}_{hotel}$  
3. Normalização dimensional: $u = \text{dot} / 10$  
4. Reprojeção: $\hat{y} = \text{clip}((u \times 4) + 1, 1, 5)$  

> **Atenção:** essa função não aplica a penalidade $\lambda$. A penalidade existe apenas em `_computar_fm` para ranqueamento; a predição para RMSE é o produto escalar puro.

O **ground truth** $y$ é a média aritmética das 10 notas da avaliação:
$$y = \frac{nota_{luxo} + nota_{lazer} + \cdots + nota_{capacidade}}{10}$$

$$\text{RMSE} = \sqrt{\frac{1}{M}\sum_{i=1}^{M}(y_i - \hat{y}_i)^2}$$

### 4.2 NDCG — Avalia o KNN
Filtra **exclusivamente** avaliações com `logica_geracao = 'KNN'` (interações orgânicas da UI onde o usuário escolheu via KNN). Dados sintéticos do `data_generator.py` possuem `logica = 'Perfil+Região+Tradeoff'` e **não entram no cálculo do NDCG**.

Para cada avaliação orgânica KNN, usa `posicao_exibicao` gravada no momento da escolha. O fallback $p = 3.0$ é aplicado apenas ao edge case de `posicao_exibicao IS NULL OR = 0`.

$$DCG_i = \frac{1}{\log_2(p_i + 1)}, \quad IDCG = \frac{1}{\log_2(1+1)} = 1.0$$

$$NDCG_i = \frac{DCG_i}{IDCG}, \quad NDCG_{global} = \frac{1}{|KNN|}\sum_i NDCG_i$$

In [ ]:
# --- Replicação exata de _obter_predicao_fm_real (recomendacao_controller.py) ---

def obter_predicao_fm_real(w_usuario_01, f_hotel_01):
    """
    Predição usada exclusivamente no cálculo de RMSE.
    - Sem penalidade Luxo×Urbano (penalidade existe apenas no ranking _computar_fm).
    - Normaliza por 10 dimensões para evitar explosão dimensional.
    """
    dot_product   = np.dot(w_usuario_01, f_hotel_01)   # produto escalar em [0, ~10]
    utilidade_norm = dot_product / 10.0                 # normaliza → [0, 1]
    predicao_escala = (utilidade_norm * 4.0) + 1.0      # projeta → [1, 5]
    return float(np.clip(predicao_escala, 1.0, 5.0))

# Simula avaliações com logica_geracao IN ('FM', 'Perfil+Região+Tradeoff')
notas_fm = np.array([
    [5, 3, 1, 1, 1, 3, 5, 1, 5, 2],  # usuário orgânico FM
    [4, 4, 2, 2, 1, 3, 4, 1, 4, 2],  # usuário sintético Perfil+Região+Tradeoff
    [5, 3, 1, 1, 1, 4, 5, 1, 5, 3],
])

# Ground truth: média das 10 notas
y_real = notas_fm.mean(axis=1)  

# Predições: para cada linha, usa perfil desse usuário (simplificação: usa a própria linha)
w_usuarios = (notas_fm - 1.0) / 4.0  # normaliza para [0,1]

erros_sq = []
for i in range(len(notas_fm)):
    pred = obter_predicao_fm_real(w_usuarios[i], hotel_exemplo)  # hotel_exemplo da Célula 1
    erro = (y_real[i] - pred) ** 2
    erros_sq.append(erro)
    print(f"Avaliação {i+1}: y_real={y_real[i]:.2f} | ŷ_FM={pred:.3f} | erro²={erro:.4f}")

rmse = math.sqrt(np.mean(erros_sq))
print(f"\nRMSE Global (FM): {rmse:.4f}")

In [ ]:
# --- Replicação exata do cálculo de NDCG (recomendacao_controller.py) ---
# Filtra APENAS logica_geracao == 'KNN' (interações orgânicas da UI)
# Dados sintéticos (Perfil+Região+Tradeoff / Aleatório) NÃO entram aqui.

# posicoes_exibicao: registradas em finalizar_com_avaliacao() para sessões KNN
posicoes_knn_organico = [1, 3, 2, 1, 5, 2]  # posicao_exibicao de avaliações KNN reais

def calcular_ndcg_global_knn(lista_posicoes):
    """
    Espelha o loop em _gerar_dashboard_metricas_globais() para df_knn.
    Assume relevância binária = 1 (o item escolhido é sempre relevante).
    IDCG fixo: item ideal sempre na posição 1.
    Fallback p=3.0 apenas para posicao_exibicao nula ou zero (edge case).
    """
    idcg = 1.0 / math.log2(1 + 1)  # = 1.0 (posição ideal = 1)
    ndcg_lista = []
    for pos in lista_posicoes:
        p = pos if (pos is not None and pos > 0) else 3.0  # edge case: posicao nula
        dcg  = 1.0 / math.log2(p + 1)
        ndcg_lista.append(dcg / idcg)

    return np.mean(ndcg_lista)

ndcg_global = calcular_ndcg_global_knn(posicoes_knn_organico)
print("Posições de exibição (sessões KNN orgânicas):", posicoes_knn_organico)
print(f"NDCG Global (KNN orgânico): {ndcg_global:.4f}")
print()
print("Interpretação:")
print(f"  NDCG = 1.000 → todos os itens escolhidos estavam na posição 1")
print(f"  NDCG = {calcular_ndcg_global_knn([3]*6):.3f} → todos na posição 3 (fallback sintético)")
print(f"  Resultado atual ({ndcg_global:.3f}) → qualidade real das sessões KNN gravadas")

## 5. Normalização de Scores para Exibição (UI)

Os scores brutos de KNN (cosseno, $[-1,1]$) e FM (produto escalar - penalidade, sem escala fixa) são heterogêneos. A função `enriquecer_e_normalizar_recomendacoes()` em `app.py` projeta ambos para a escala $[1, 5]$ (campo `afinidade`) usando **min-max normalização** com lógica invertida para KNN:

- **KNN** (maior similaridade = melhor, score mais alto): `afinidade = ((s_max - score) / (s_max - s_min)) * 4 + 1`  
  *(inversão: score máximo → afinidade = 1; score mínimo → afinidade = 5)*

- **FM** (maior utilidade = melhor): `afinidade = ((score - s_min) / (s_max - s_min)) * 4 + 1`

> Essa normalização é **apenas para exibição** na UI e não afeta os cálculos internos de RMSE e NDCG.

### Referências Acadêmicas
- **Pazzani & Billsus (2007)**: Sistemas Baseados em Conteúdo (base para a vetorização de 10 atributos de hotéis).  
- **Rendle (2010)**: Factorization Machines (referência conceitual do produto escalar com interações latentes).  
- **Järvelin & Kekäläinen (2002)**: Avaliação Cumulativa (DCG e NDCG aplicados na mensuração de ranking da UI).